# 05 pipeline Versuche

Wichtiges aus der EDA:

|EDA Erkenntnis|Feature|Preproc Entscheidung|Begründung|
|---|---|---|---|
|Hohe Kardinalität|ps_car_11_cat|Target Encoding|bei 104 Klassen wäre one hot extremer bloat|
|Viele MVs, MV zeigt Signal für target|ps_car_03_cat, ps_car_05_cat, ps_reg_03|Missing feature binär neben imputieren, bei xgboost nan|Wenn imputiert/dropped, könnte signal verlieren|
|Mögliches eigene Gruppe|ps_ind_[16,17,18]_bin|erst übernehmen, bei feature engineering untersuchen||
|Redundanzen|ps_ind_12_bin, ps_ind_14|Erst beibehalten, drop test in modellvergleichen||
|Calc Gruppe kaum lineare korr mit target|ps_calc_*|Pipelines mit und ohne calc, modellvergleich später||
|Starke Klassenimbalance in target|target|Kein Resampling in preproc aber in modell training könnten wir oversampling oä versuchen, modelle nicht über acc sondern wie kaggle über gini bewerten||
|verschiedene Bereiche Verteilungen der continous features|ps_reg_*, ps_car_1[1-5], ps_calc_0[1-3]|Standard scaler für lineare modelle||

Weiteres:
- XGBoost verarbeitet nan nativ, also für xgboost keine imputation

## Imports

In [13]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.model_selection import KFold

import matplotlib as mpl
import matplotlib.pyplot as plt

In [2]:
mpl.style.use("seaborn-v0_8-colorblind")

In [3]:
RANDOM_STATE = 42

## Dataframe und Splits

In [4]:
df_raw = fetch_openml(data_id=42742, as_frame=True).frame

In [5]:
feat_cols = [col for col in df_raw.columns if col not in ("target",)]
cat_cols = [col for col in feat_cols if col.endswith("_cat")]
bin_cols = [col for col in feat_cols if col.endswith("_bin")]
num_cols = [col for col in feat_cols if col not in cat_cols and col not in bin_cols]

In [6]:
for col in cat_cols:
    df_raw[col] = df_raw[col].astype("category")

for col in bin_cols:
    if df_raw[col].isna().sum() == 0:
        df_raw[col] = df_raw[col].astype(int).astype("bool")
    else:
        df_raw[col] = df_raw[col].astype("Int8")

for col in num_cols:
    df_raw[col] = df_raw[col].astype("float32")

df_raw["target"] = df_raw["target"].astype(int).astype("bool")

In [7]:
idx_train = np.load("../../../data/processed/train_idx.npy")
idx_val = np.load("../../../data/processed/val_idx.npy")
idx_test = np.load("../../../data/processed/test_idx.npy")

df_train = df_raw.iloc[idx_train]
df_val = df_raw.iloc[idx_val]
df_test = df_raw.iloc[idx_test]

In [8]:
x_train, y_train = df_train[feat_cols], df_train["target"]
x_val, y_val = df_val[feat_cols], df_val["target"]
x_test, y_test = df_test[feat_cols], df_test["target"]

pd.Series(
    {
        "train": [len(df_train), len(x_train), len(y_train)],
        "val": [len(df_val), len(x_val), len(y_val)],
        "test": [len(df_test), len(x_test), len(y_test)]
    }
)

train    [476168, 476168, 476168]
val         [59522, 59522, 59522]
test        [59522, 59522, 59522]
dtype: object

## First pipelines

In [9]:
# Hilfsvariablen

calc_cols = [c for c in feat_cols if c.startswith("ps_calc_")]

mv_cols = ["ps_car_03_cat", "ps_car_05_cat", "ps_reg_03", "ps_car_14"]

num_cols_no_calc = [c for c in num_cols if c not in calc_cols]
bin_cols_no_calc = [c for c in bin_cols if c not in calc_cols]

high_kard_cols = ["ps_car_11_cat"]
low_kard_cols = [c for c in cat_cols if c not in high_kard_cols]


In [10]:
pd.Series({
    "feat_cols (df_raw)": len(feat_cols),
    "in Pipeline (Summe Hilfsvariablen)": len(num_cols) + len(low_kard_cols) + len(high_kard_cols) + len(bin_cols),
})

feat_cols (df_raw)                    57
in Pipeline (Summe Hilfsvariablen)    57
dtype: int64

## Modelle für die prepared werden soll
- logistische Regression
- XGBoost
- Random Forest(?)

Für jedes der Modelle sammle ich hier wichtige Punkte für die Preproc und erstelle zunächst eine Testpipeline zur Vorstellung

## Logistische Regression
- Nur numerische Daten: onehot und target encoder für cat cols
- Keine nan: SimpleImputer
- Skalierung: Standardscaler (Robustscaler?)
- Multikollinearität ausschließen: drop or regularisierung im training?

Plan: num_cols: impute&indicator-->scale; cat_low_kard-->onehot; cat_high_kard-->target; bin so lassen, rest droppen

Rssourcen (Aufrufdatum: 04.08.2026):
- https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html
- https://www.geeksforgeeks.org/machine-learning/what-is-exactly-sklearnpipelinepipeline/
- https://scikit-learn.org/stable/modules/preprocessing.html
- https://scikit-learn.org/stable/modules/impute.html
- https://medium.com/analytics-vidhya/machine-learning-ii-logistic-regression-explained-data-pre-processing-hands-on-kaggle-728e6a9d4bbf
https://medium.com/my-data-camp-journey/preprocessing-data-for-logistic-regression-f311c937d765
- https://github.com/NaflanNadeer/Machine-Learning-with-Logistic-Regression-and-Data-Preprocessing


In [11]:
preproc_logreg = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler())
    ]), num_cols),
    ("cat_onehot", OneHotEncoder(handle_unknown="ignore"), low_kard_cols),
    ("cat_target", TargetEncoder(random_state=RANDOM_STATE), high_kard_cols),
    ("bin", "passthrough", bin_cols)
], remainder="drop")

In [12]:
preproc_logreg.fit(x_train, y_train)
x_train_logreg = preproc_logreg.transform(x_train)
x_train_logreg.shape

c:\Users\Linus Lauschke\anaconda3\envs\ADA\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


(476168, 127)

In [14]:
# version mit kfold für target encoder (bessere Reproduzierbarkeit)

cv_targetenc = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

preproc_logregMK2 = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler())
    ]), num_cols),
    ("cat_onehot", OneHotEncoder(handle_unknown="ignore"), low_kard_cols),
    ("cat_target", TargetEncoder(cv=cv_targetenc), high_kard_cols),
    ("bin", "passthrough", bin_cols)
], remainder="drop")

preproc_logregMK2.fit(x_train, y_train)
x_train_logregMK2 = preproc_logregMK2.transform(x_train)
x_train_logregMK2.shape

(476168, 127)

### Sanity Checks logreg preprocessing

In [15]:
type(x_train_logreg), type(x_train_logregMK2)

(numpy.ndarray, numpy.ndarray)

In [22]:
pd.Series({
    "nan MK1" : np.isnan(x_train_logreg).sum(),
    "nan MK2" : np.isnan(x_train_logregMK2).sum(),
    "Spalten MK1" : x_train_logreg.shape[1],
    "Spalten MK2" : x_train_logregMK2.shape[1],
    "Zeilen MK1" : x_train_logreg.shape[0],
    "Zeilen MK2" : x_train_logregMK2.shape[0]
})

nan MK1             0
nan MK2             0
Spalten MK1       127
Spalten MK2       127
Zeilen MK1     476168
Zeilen MK2     476168
dtype: int64

## XGBoost
- Kategorische Daten: nativ unterstützt, kein encoding
- nan: kein imputing nötig
- scaling: für bäume nicht nötig

möglich:
- encodings versuchen für besseren vergleich zu logreg als Modell?
- calc Gruppe droppen?

Plan: raw, also kein preprocess eig

Ressourcen (Aufrufdatum 05.08.2026):
- https://xgboost.readthedocs.io/en/stable/
    - https://xgboost.readthedocs.io/en/stable/tutorials/categorical.html
- https://developer.nvidia.com/blog/categorical-features-in-xgboost-without-manual-encoding/
- https://xgboosting.com/xgboosts-native-support-for-categorical-features/

## Notes to self
- add indicator um mv spalten anzuhängen
- simpleimpute mit median, weniger anfällig für ausreißer (vllt KNNimputer versuchen, aber bei vielen MV eventuell ungenau)
- hilfreich für später?: https://scikit-learn.org/stable/auto_examples/compose/plot_digits_pipe.html
- da transformer array ausgibt muss später feature mit prepro.get_feature_names_out() und koeffizient mit model.coef_[0] angesehen werden (also als df verknüpfen) ; in zukunft aber eh in einer pipeline
